# 02. Feature Engineering

Build features for market stress classification: portfolio volatility, VIX, yield spread, momentum.

Stress periods are defined as days when realized portfolio volatility exceeds the 75th percentile.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from features import build_feature_matrix

## Load prices and build features

In [ ]:
prices = pd.read_csv('../data/raw/prices.csv', index_col=0, parse_dates=True)
print(f"Prices shape: {prices.shape}")

features = build_feature_matrix(prices)
print(f"Features shape: {features.shape}")
print(f"\nColumns: {list(features.columns)}")

## Stress regime distribution

In [ ]:
stress_counts = features['stress'].value_counts().sort_index()
print("Stress label distribution:")
print(f"  Normal (0): {stress_counts[0]} days ({stress_counts[0]/len(features)*100:.1f}%)")
print(f"  Stress (1): {stress_counts[1]} days ({stress_counts[1]/len(features)*100:.1f}%)")

imbalance_ratio = stress_counts[0] / stress_counts[1]
print(f"\nClass imbalance ratio (normal:stress): {imbalance_ratio:.1f}:1")
print("\nNote: Class imbalance is moderate. Threshold metrics (precision, recall) will be important.")

## Time-series visualization

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 10))

for ax, col in zip(axes, ['portfolio_vol', 'vix', 'yield_spread', 'momentum']):
    ax.plot(features.index, features[col], label=col, color='steelblue', linewidth=0.8)
    
    # Shade stress periods
    stress_idx = features[features['stress'] == 1].index
    ax.axvspan(stress_idx.min(), stress_idx.max(), alpha=0.1, color='red', label='Stress region')
    
    ax.set_ylabel(col)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper left', fontsize=8)

axes[-1].set_xlabel('Date')
fig.tight_layout()
plt.savefig('../figures/02_feature_timeseries.png', dpi=100, bbox_inches='tight')
plt.show()

print("Saved figure: figures/02_feature_timeseries.png")

## Summary statistics

In [ ]:
print("Feature summary statistics:")
print(features.describe())

## Save features

In [ ]:
features.to_csv('../data/processed/features.csv')
print(f"Saved to data/processed/features.csv ({features.shape[0]} rows × {features.shape[1]} columns)")